In [8]:
# Exploración de la API de eBird
#
# La API usa códigos jerárquicos de región:
# - world devuelve países.
# - Un país, por ejemplo CL, devuelve sus regiones administrativas.
# - Una región, por ejemplo CL-RM, puede devolver sus subregiones.
#
# Las observaciones recientes se consultan con /data/obs/{regionCode}/recent.
# La API no entrega directamente archivos de foto o audio; hasRichMedia
# indica que puede existir multimedia asociada a la observación.

## Qué entrega esta consulta

Cada observación puede incluir especie (`comName`, `sciName`), fecha (`obsDt`), lugar (`locName`), cantidad (`howMany`), coordenadas (`lat`, `lng`) y, según el endpoint y la observación, `hasRichMedia`.

Para ordenar las aves más comunes habría que agrupar por `comName` y contar observaciones o sumar `howMany`. Para fotos y cantos habrá que consultar una fuente multimedia aparte; esta API solo permite detectar que puede existir contenido enriquecido, no descargarlo directamente.

In [7]:
# Resumen útil para una futura visualización.
import pandas as pd

observations_df = pd.DataFrame(observations)
summary_columns = [
    "comName",
    "sciName",
    "obsDt",
    "locName",
    "howMany",
    "lat",
    "lng",
    "hasRichMedia",
]
summary = observations_df.reindex(columns=summary_columns)
summary.head(20)

,comName,sciName,obsDt,locName,howMany,lat,lng,hasRichMedia
0,Diuca Finch,Diuca diuca,2026-09-17 12:43,Parquemet--Cerro San Cristóbal--Sector Tupahue,2,-33.415886,-70.622736,NaN
1,Long-tailed Meadowlark,Leistes loyca,2026-09-17 12:43,Parquemet--Cerro San Cristóbal--Sector Tupahue,3,-33.415886,-70.622736,NaN
2,Austral Blackbird,Curaeus curaeus,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",2,-33.400910,-71.229271,NaN
3,White-crested Elaenia,Elaenia albiceps,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",2,-33.400910,-71.229271,NaN
4,Chilean Mockingbird,Mimus thenca,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",6,-33.400910,-71.229271,NaN
5,California Quail,Callipepla californica,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",7,-33.400910,-71.229271,NaN
6,Rufous-collared Sparrow,Zonotrichia capensis,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",8,-33.400910,-71.229271,NaN
7,Chimango Caracara,Daptrius chimango,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",1,-33.400910,-71.229271,NaN
8,Southern House Wren,Troglodytes musculus,2026-09-17 12:28,"Curacavi, Santiago Metropolitan Region, CL (-3...",1,-33.400910,-71.229271,NaN
9,Shiny Cowbird,Molothrus bonariensis,2026-09-17 12:18,Parque Bicentenario de Cerrillos,19,-33.493379,-70.699511,NaN


In [9]:
common_species = (
    observations_df.groupby(["speciesCode", "comName", "sciName"], dropna=False)
    .agg(
        observaciones=("speciesCode", "size"),
        individuos_reportados=("howMany", "sum"),
        lugares=("locId", "nunique"),
        ultima_observacion=("obsDt", "max"),
    )
    .sort_values(["observaciones", "individuos_reportados"], ascending=False)
    .reset_index()
)

common_species.head(20)

,speciesCode,comName,sciName,observaciones,individuos_reportados,lugares,ultima_observacion
0,eardov1,Eared Dove,Zenaida auriculata,1,46,1,2026-09-17 12:18
1,gryfin2,Greater Yellow-Finch,Sicalis auriventris,1,45,1,2026-09-16 16:22
2,andgoo1,Andean Goose,Oressochen melanopterus,1,41,1,2026-09-16 09:42
3,brhgul2,Brown-hooded Gull,Chroicocephalus maculipennis,1,32,1,2026-09-16 09:42
4,yebpin1,Yellow-billed Pintail,Anas georgica,1,26,1,2026-09-16 10:02
5,bknsti,Black-necked Stilt,Himantopus mexicanus,1,24,1,2026-09-16 09:42
6,gryfin1,Grassland Yellow-Finch,Sicalis luteola,1,24,1,2026-09-17 12:18
7,categr1,Western Cattle-Egret,Ardea ibis,1,20,1,2026-09-16 09:42
8,shicow,Shiny Cowbird,Molothrus bonariensis,1,19,1,2026-09-17 12:18
9,regcoo1,Red-gartered Coot,Fulica armillata,1,12,1,2026-09-16 11:27


In [6]:
# Consulta de observaciones recientes de una región.
# Cambia este código por cualquiera de chile_regions.
REGION_CODE = "CL-RM"

observations = ebird_get(
    f"/data/obs/{REGION_CODE}/recent",
    params={
        "back": 7,
        "maxResults": 100,
        "detail": "full",
        "includeProvisional": "true",
    },
)

print(f"Observaciones recibidas para {REGION_CODE}: {len(observations)}")
observations[:3]

Observaciones recibidas para CL-RM: 100


[{'speciesCode': 'codfin1',
  'comName': 'Diuca Finch',
  'sciName': 'Diuca diuca',
  'locId': 'L9990332',
  'locName': 'Parquemet--Cerro San Cristóbal--Sector Tupahue',
  'obsDt': '2026-09-17 12:43',
  'howMany': 2,
  'lat': -33.4158865,
  'lng': -70.6227357,
  'obsValid': True,
  'obsReviewed': False,
  'locationPrivate': False,
  'subId': 'S393688912'},
 {'speciesCode': 'lotmea1',
  'comName': 'Long-tailed Meadowlark',
  'sciName': 'Leistes loyca',
  'locId': 'L9990332',
  'locName': 'Parquemet--Cerro San Cristóbal--Sector Tupahue',
  'obsDt': '2026-09-17 12:43',
  'howMany': 3,
  'lat': -33.4158865,
  'lng': -70.6227357,
  'obsValid': True,
  'obsReviewed': False,
  'locationPrivate': False,
  'subId': 'S393688912'},
 {'speciesCode': 'ausbla1',
  'comName': 'Austral Blackbird',
  'sciName': 'Curaeus curaeus',
  'locId': 'L77312056',
  'locName': 'Curacavi, Santiago Metropolitan Region, CL (-33.401, -71.229)',
  'obsDt': '2026-09-17 12:28',
  'howMany': 2,
  'lat': -33.4009098,
  'l

In [5]:
# Regiones administrativas de Chile.
chile_regions = show_region_options("subnational1", "CL")
chile_regions

[{'code': 'CL-AI', 'name': 'Aisén del General Carlos Ibáñez del Campo'},
 {'code': 'CL-AN', 'name': 'Antofagasta'},
 {'code': 'CL-AR', 'name': 'Araucanía'},
 {'code': 'CL-AP', 'name': 'Arica y Parinacota'},
 {'code': 'CL-AT', 'name': 'Atacama'},
 {'code': 'CL-BI', 'name': 'Bío-Bío'},
 {'code': 'CL-CO', 'name': 'Coquimbo'},
 {'code': 'CL-LI', 'name': "Libertador General Bernardo O'Higgins"},
 {'code': 'CL-LL', 'name': 'Los Lagos'},
 {'code': 'CL-LR', 'name': 'Los Ríos'},
 {'code': 'CL-MA', 'name': 'Magallanes'},
 {'code': 'CL-ML', 'name': 'Maule'},
 {'code': 'CL-RM', 'name': 'Región Metropolitana de Santiago'},
 {'code': 'CL-TA', 'name': 'Tarapacá'},
 {'code': 'CL-VS', 'name': 'Valparaíso'},
 {'code': 'CL-NB', 'name': 'Ñuble'}]

In [3]:
def ebird_get(path, params=None):
    response = requests.get(
        f"{BASE_URL}{path}",
        headers={"X-eBirdApiToken": EBIRD_API_KEY},
        params=params,
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


def show_region_options(region_type, parent_code):
    """List regions below a world, country, or subnational1 parent."""
    return ebird_get(f"/ref/region/list/{region_type}/{parent_code}")


# Países disponibles para consultar.
world_regions = show_region_options("country", "world")
world_regions[:10]

[{'code': 'AF', 'name': 'Afghanistan'},
 {'code': 'AL', 'name': 'Albania'},
 {'code': 'DZ', 'name': 'Algeria'},
 {'code': 'AS', 'name': 'American Samoa'},
 {'code': 'AD', 'name': 'Andorra'},
 {'code': 'AO', 'name': 'Angola'},
 {'code': 'AI', 'name': 'Anguilla'},
 {'code': 'AQ', 'name': 'Antarctica'},
 {'code': 'AG', 'name': 'Antigua and Barbuda'},
 {'code': 'AR', 'name': 'Argentina'}]

In [1]:
from pathlib import Path
import os

import requests

BASE_URL = "https://api.ebird.org/v2"


def load_env_value(name):
    """Load one simple KEY=value entry without printing the secret."""
    value = os.getenv(name)
    if value:
        return value

    env_path = Path.cwd() / ".env"
    if env_path.exists():
        for line in env_path.read_text(encoding="utf-8").splitlines():
            key, separator, candidate = line.partition("=")
            if separator and key.strip() == name:
                return candidate.strip().strip('"').strip("'")
    return None


EBIRD_API_KEY = (
    load_env_value("EBIRD_API_KEY")
    or load_env_value("API_BIRD_KEY")
    or load_env_value("X_EBIRDAPITOKEN")
)

if not EBIRD_API_KEY:
    raise RuntimeError("No se encontró EBIRD_API_KEY, API_BIRD_KEY o X_EBIRDAPITOKEN en .env")

print("Clave cargada: sí")
print("Directorio de trabajo:", Path.cwd())

Clave cargada: sí
Directorio de trabajo: /Users/jabac/Documents/universidad/vi-semestre/,infovis/proyecto


# Prototipo interactivo: aves por región de Chile

Este prototipo reutiliza los avistamientos históricos del dataset y los agrega por mes y región. El mapa muestra la participación relativa de cada región en el total de observaciones; al seleccionar una región se actualiza un panel con sus cinco especies más reportadas. Las imágenes y audios quedan como espacios reservados para una segunda fuente multimedia.

In [11]:
from pathlib import Path
from datetime import date, timedelta
import os
import html
import unicodedata

import geopandas as gpd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import HTML, display

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "Regiones" / "Regional.shp").exists():
    PROJECT_DIR = Path("/Users/jabac/Documents/universidad/vi-semestre/,infovis/proyecto")

SHAPEFILE_PATH = PROJECT_DIR / "Regiones" / "Regional.shp"
DATA_CANDIDATES = [
    PROJECT_DIR / "obs_2022-2025_final.parquet",
    PROJECT_DIR / "data" / "T2" / "obs_2022-2025_final.parquet",
    Path.cwd() / "obs_2022-2025_final.parquet",
]

if "observations_df" in globals() and not observations_df.empty:
    prototype_df = observations_df.copy()
elif "df" in globals() and not df.empty:
    prototype_df = df.copy()
else:
    parquet_path = next((path for path in DATA_CANDIDATES if path.exists()), None)
    if parquet_path is None:
        raise FileNotFoundError(
            "No se encontró un DataFrame de observaciones ni un parquet. "
            "Ejecuta primero la extracción o deja el parquet en el proyecto."
        )
    prototype_df = pd.read_parquet(parquet_path)

chile_map = gpd.read_file(SHAPEFILE_PATH).to_crs(4326)
map_name_column = next(
    (
        column
        for column in ("NOM_REG", "NOMBRE", "REGION", "Region", "NOM_REGION", "NAME_1")
        if column in chile_map.columns
    ),
    None,
)
if map_name_column is None:
    raise KeyError(f"No se encontró la columna de nombre regional. Columnas: {list(chile_map.columns)}")

region_names = {
    "CL-AI": "Aisén del General Carlos Ibáñez del Campo",
    "CL-AN": "Antofagasta",
    "CL-AR": "Araucanía",
    "CL-AP": "Arica y Parinacota",
    "CL-AT": "Atacama",
    "CL-BI": "Bío-Bío",
    "CL-CO": "Coquimbo",
    "CL-LI": "Libertador General Bernardo O'Higgins",
    "CL-LL": "Los Lagos",
    "CL-LR": "Los Ríos",
    "CL-MA": "Magallanes",
    "CL-ML": "Maule",
    "CL-RM": "Región Metropolitana de Santiago",
    "CL-TA": "Tarapacá",
    "CL-VS": "Valparaíso",
    "CL-NB": "Ñuble",
}


def normalize_text(value):
    normalized = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode()
    return " ".join(normalized.upper().replace("-", " ").split())


name_to_code = {normalize_text(name): code for code, name in region_names.items()}
numeric_region_codes = {
    1: "CL-AP",
    2: "CL-TA",
    3: "CL-AN",
    4: "CL-AT",
    5: "CL-CO",
    6: "CL-VS",
    7: "CL-RM",
    8: "CL-LI",
    9: "CL-ML",
    10: "CL-BI",
    11: "CL-AR",
    12: "CL-LL",
    13: "CL-LR",
    14: "CL-AI",
    15: "CL-MA",
    16: "CL-NB",
}
region_column = next(
    (column for column in ("subnational1Code", "regionCode", "region_code") if column in prototype_df),
    None,
)

prototype_df = prototype_df.copy()
if region_column is not None:
    prototype_df["region_code"] = prototype_df[region_column].astype(str)
elif {"lat", "lng"}.issubset(prototype_df.columns):
    points = gpd.GeoDataFrame(
        prototype_df,
        geometry=gpd.points_from_xy(prototype_df["lng"], prototype_df["lat"]),
        crs="EPSG:4326",
    )
    spatial = gpd.sjoin(
        points,
        chile_map[[map_name_column, "geometry"]],
        how="left",
        predicate="within",
    )
    prototype_df["region_code"] = spatial[map_name_column].map(
        lambda value: name_to_code.get(normalize_text(value))
    )
else:
    raise KeyError("El dataset necesita subnational1Code o las columnas lat y lng.")

prototype_df["date"] = pd.to_datetime(prototype_df["obsDt"].astype(str).str[:10], errors="coerce")
prototype_df["year_month"] = prototype_df["date"].dt.to_period("M").astype(str)
prototype_df["howMany"] = pd.to_numeric(prototype_df.get("howMany", 1), errors="coerce").fillna(1)
prototype_df = prototype_df.dropna(subset=["date", "region_code"])

print(f"Observaciones disponibles: {len(prototype_df):,}")
print(f"Cobertura: {prototype_df.date.min():%Y-%m-%d} a {prototype_df.date.max():%Y-%m-%d}")
print(f"Regiones observadas: {prototype_df.region_code.nunique()}")

Observaciones disponibles: 100
Cobertura: 2026-09-14 a 2026-09-17
Regiones observadas: 1


## Extracción histórica opcional

eBird no ofrece un endpoint único que devuelva directamente la serie mensual por región y especie. El endpoint histórico consulta un día a la vez; por eso la estrategia es descargar observaciones diarias, guardarlas en caché y agregarlas después por mes. Diez años implican aproximadamente 3.650 solicitudes, así que esta celda no se ejecuta automáticamente.

In [ ]:
import json
import time

API_URL = "https://api.ebird.org/v2/data/obs/CL/historic"
CACHE_DIR = PROJECT_DIR / "cache_ebird"
CACHE_DIR.mkdir(exist_ok=True)


def read_api_key():
    env_path = PROJECT_DIR / ".env"
    for line in env_path.read_text(encoding="utf-8").splitlines():
        key, separator, value = line.partition("=")
        if separator and key.strip() in {"API_BIRD_KEY", "EBIRD_API_KEY", "X_EBIRDAPITOKEN"}:
            return value.strip().strip('"').strip("'")
    raise RuntimeError("No se encontró la API key en .env")


def fetch_historic_daily(start_date, end_date, pause_seconds=0.15):
    """Descarga y cachea observaciones diarias para agregarlas después por mes."""
    import requests

    api_key = read_api_key()
    rows = []
    current = start_date
    while current <= end_date:
        cache_file = CACHE_DIR / f"{current:%Y-%m-%d}.json"
        if cache_file.exists():
            daily_rows = json.loads(cache_file.read_text(encoding="utf-8"))
        else:
            response = requests.get(
                f"{API_URL}/{current:%Y/%m/%d}",
                headers={"X-eBirdApiToken": api_key},
                params={"sppLocale": "es_CL"},
                timeout=60,
            )
            response.raise_for_status()
            daily_rows = response.json()
            cache_file.write_text(json.dumps(daily_rows), encoding="utf-8")
            time.sleep(pause_seconds)
        rows.extend(daily_rows)
        current += timedelta(days=1)
    return pd.DataFrame(rows)


# Para solicitar la década completa, cambia a True y ejecuta explícitamente.
FETCH_TEN_YEARS = False
TEN_YEAR_START = date.today().replace(year=date.today().year - 10)
TEN_YEAR_END = date.today()

# historic_df = fetch_historic_daily(TEN_YEAR_START, TEN_YEAR_END)
# historic_df.to_parquet(PROJECT_DIR / "obs_10_years.parquet", index=False)

In [14]:
# El shapefile usa códigos administrativos históricos: 15=Arica, 1=Tarapacá, ..., 16=Ñuble.
numeric_region_codes = {
    15: "CL-AP", 1: "CL-TA", 2: "CL-AN", 3: "CL-AT", 4: "CL-CO", 5: "CL-VS",
    13: "CL-RM", 6: "CL-LI", 7: "CL-ML", 8: "CL-BI", 9: "CL-AR", 10: "CL-LL",
    14: "CL-LR", 11: "CL-AI", 12: "CL-MA", 16: "CL-NB",
}

chile_map["region_code"] = pd.to_numeric(chile_map["codregion"], errors="coerce").map(numeric_region_codes)

region_metrics = (
    prototype_df.groupby("region_code")
    .agg(
        observaciones=("speciesCode", "size"),
        individuos_reportados=("howMany", "sum"),
        especies=("speciesCode", "nunique"),
        meses=("year_month", "nunique"),
    )
    .reset_index()
)
region_metrics["participacion"] = region_metrics["observaciones"] / region_metrics["observaciones"].sum() * 100
map_metrics = chile_map.merge(region_metrics, on="region_code", how="left").fillna(
    {"observaciones": 0, "individuos_reportados": 0, "especies": 0, "meses": 0, "participacion": 0}
)
map_metrics[["region_code", map_name_column, "observaciones", "participacion", "especies"]]

,region_code,Region,observaciones,participacion,especies
0,CL-AP,Región de Arica y Parinacota,0.0,0.0,0.0
1,CL-TA,Región de Tarapacá,0.0,0.0,0.0
2,CL-AN,Región de Antofagasta,0.0,0.0,0.0
3,CL-MA,Región de Magallanes y Antártica Chilena,0.0,0.0,0.0
4,CL-AI,Región de Aysén del Gral.Ibañez del Campo,0.0,0.0,0.0
5,CL-AT,Región de Atacama,0.0,0.0,0.0
6,CL-CO,Región de Coquimbo,0.0,0.0,0.0
7,CL-VS,Región de Valparaíso,0.0,0.0,0.0
8,CL-RM,Región Metropolitana de Santiago,100.0,100.0,100.0
9,CL-LL,Región de Los Lagos,0.0,0.0,0.0


In [ ]:
## Visualización web

La visualización interactiva no se ejecuta dentro del notebook. La página está en `web/index.html` y consume los archivos `web/data/regions.geojson`, `web/data/observations.json` y `web/data/metadata.json` generados por la celda final. Esto permite publicar el frontend en GitHub Pages sin exponer la API key.

ImportError: Please install anywidget to use the FigureWidget class

In [ ]:
La serie mensual se representa en la webpage. El notebook conserva `year_month` para permitir nuevas exportaciones y análisis.

## Exportar datos para la webpage

La página web no ejecuta Python ni expone la API key. Esta celda publica los datos procesados como archivos estáticos que el frontend puede leer.

In [17]:
import json

WEB_DATA_DIR = PROJECT_DIR / "web" / "data"
WEB_DATA_DIR.mkdir(parents=True, exist_ok=True)

export_columns = [
    "speciesCode", "comName", "sciName", "obsDt", "howMany",
    "locId", "locName", "lat", "lng", "region_code", "year_month",
]
export_observations = prototype_df.reindex(columns=export_columns).copy()
export_observations = export_observations.where(export_observations.notna(), None)

map_metrics.to_file(WEB_DATA_DIR / "regions.geojson", driver="GeoJSON")
(WEB_DATA_DIR / "observations.json").write_text(
    json.dumps(export_observations.to_dict(orient="records"), ensure_ascii=False, default=str),
    encoding="utf-8",
)
(WEB_DATA_DIR / "metadata.json").write_text(
    json.dumps(
        {
            "updatedAt": pd.Timestamp.now(tz="UTC").isoformat(),
            "startDate": prototype_df["date"].min().strftime("%Y-%m-%d"),
            "endDate": prototype_df["date"].max().strftime("%Y-%m-%d"),
            "observationCount": int(len(prototype_df)),
            "regionCount": int(prototype_df["region_code"].nunique()),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Datos exportados en {WEB_DATA_DIR}")

Datos exportados en /Users/jabac/Documents/universidad/vi-semestre/,infovis/proyecto/web/data
